In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/anonymous111111111/doha-dataset/dohas_final_hindi_dataset.csv


In [2]:
!pip install -q transformers datasets peft pandas accelerate sentencepiece protobuf

In [3]:
import pandas as pd
from datasets import Dataset
import os

# Update this path to match your uploaded file's path!
csv_path = "/kaggle/input/datasets/anonymous111111111/doha-dataset/dohas_final_hindi_dataset.csv" 
df = pd.read_csv(csv_path)

# Clean and format
df = df[df['Doha'] != 'Doha'] 
df = df.dropna(subset=['Theme', 'Context', 'Doha'])

df["input_text"] = "Generate Hindi Doha | Theme: " + df["Theme"].astype(str) + " | Context: " + df["Context"].astype(str)
df["target_text"] = df["Doha"].astype(str)
df = df[["input_text", "target_text"]]

# Convert and split
dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.1)

# Save to Kaggle's working directory
dataset.save_to_disk("/kaggle/working/doha_hf_dataset")
print(f"Dataset prepared! Total training rows: {len(dataset['train'])}")

Saving the dataset (0/1 shards):   0%|          | 0/7371 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/820 [00:00<?, ? examples/s]

Dataset prepared! Total training rows: 7371


In [4]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

# 1. Specify the base model directly from Hugging Face
model_name = "google/mt5-small"

print(f"Loading {model_name} from Hugging Face...")

# 2. Use the 'Auto' classes to prevent import errors
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# 3. Automatically use the Kaggle GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"Model loaded successfully on: {device}")

# 4. Hardcode a Theme and Context from your dataset
theme = "भक्ति"
context = "ईश्वर के प्रति समर्पण"

# 5. Format the prompt exactly as it will be formatted during training
input_text = f"Generate Hindi Doha | Theme: {theme} | Context: {context}"

print("\n" + "="*50)
print(f"Input Prompt: {input_text}")
print("="*50)

# 6. Convert the text into tokens
input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to(device)

# 7. Ask the raw model to generate an output
print("Generating baseline output...\n")
outputs = model.generate(
    input_ids,
    max_length=60,
    num_beams=5,
    early_stopping=True
)

# 8. Decode the tokens back into Hindi text
baseline_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("-" * 40)
print("Untrained Baseline Output:")
print(baseline_output)
print("-" * 40)

Loading google/mt5-small from Hugging Face...


config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded successfully on: cuda

Input Prompt: Generate Hindi Doha | Theme: भक्ति | Context: ईश्वर के प्रति समर्पण
Generating baseline output...

----------------------------------------
Untrained Baseline Output:
<extra_id_0>
----------------------------------------


In [5]:
from datasets import load_from_disk
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
from peft import get_peft_model, LoraConfig, TaskType
import os

print("1. Loading base model and tokenizer...")
model_name = "google/mt5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("2. Applying LoRA for efficient training...")
# This freezes the main model and only trains a small adapter, saving massive amounts of memory
lora_config = LoraConfig(
    r=8, 
    lora_alpha=32, 
    target_modules=["q", "v"], 
    lora_dropout=0.05, 
    task_type=TaskType.SEQ_2_SEQ_LM
)
model = get_peft_model(model, lora_config)

print("3. Loading prepared dataset...")
# Make sure you ran the prep_data step earlier so this folder exists!
if not os.path.exists("/kaggle/working/doha_hf_dataset"):
    raise FileNotFoundError("Dataset not found! Please run the Data Preparation cell first.")
    
dataset = load_from_disk("/kaggle/working/doha_hf_dataset")

print("4. Tokenizing the text for the model...")
def preprocess_function(examples):
    inputs = tokenizer(examples["input_text"], max_length=128, truncation=True, padding="max_length")
    targets = tokenizer(examples["target_text"], max_length=128, truncation=True, padding="max_length")
    # In sequence-to-sequence models, the target tokens are called "labels"
    inputs["labels"] = targets["input_ids"]
    return inputs

tokenized_datasets = dataset.map(preprocess_function, batched=True)

print("5. Setting up Training Arguments...")
training_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/doha_model_output",
    eval_strategy="epoch",           # FIXED: Updated for the latest transformers version
    learning_rate=3e-4,              # How fast the model learns
    per_device_train_batch_size=8,   # Number of dohas processed at once
    per_device_eval_batch_size=8,
    num_train_epochs=3,              # Go through the dataset 3 times
    save_strategy="epoch",
    fp16=True,                       # Turn on Mixed Precision to make Kaggle GPUs run much faster
    logging_steps=100,               # Print updates every 100 steps
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

print("6. STARTING TRAINING (This will take some time depending on the GPU)...")

trainer.train()

print("7. Training Complete! Saving the fine-tuned model...")
model.save_pretrained("/kaggle/working/final_doha_model")
tokenizer.save_pretrained("/kaggle/working/final_doha_model")
print("All done! Your custom Doha Generator is saved in /kaggle/working/final_doha_model")

1. Loading base model and tokenizer...


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


2. Applying LoRA for efficient training...
3. Loading prepared dataset...
4. Tokenizing the text for the model...


Map:   0%|          | 0/7371 [00:00<?, ? examples/s]

...

5. Setting up Training Arguments...
6. STARTING TRAINING (This will take some time depending on the GPU)...


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss
1,13.066737,8.781428
2,5.063021,3.601368
3,4.356650,3.319977


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


7. Training Complete! Saving the fine-tuned model...
All done! Your custom Doha Generator is saved in /kaggle/working/final_doha_model


In [6]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import PeftModel
import torch

print("1. Loading the newly trained model...")
# Path to where Kaggle just saved your trained weights
model_path = "/kaggle/working/final_doha_model"
base_model_name = "google/mt5-small"

# Load the tokenizer and the base model
tokenizer = AutoTokenizer.from_pretrained(model_path)
base_model = AutoModelForSeq2SeqLM.from_pretrained(base_model_name)

# MAGIC HAPPENS HERE: Attach your fine-tuned LoRA weights to the base model
model = PeftModel.from_pretrained(base_model, model_path)

# Move to GPU for fast generation
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print("Model loaded and ready!\n")

# 2. Set the exact same test prompt you used for the baseline
theme = "भक्ति"
context = "ईश्वर के प्रति समर्पण"
input_text = f"Generate Hindi Doha | Theme: {theme} | Context: {context}"

print("="*50)
print(f"Input Prompt: {input_text}")
print("="*50)

# 3. Generate the Doha
input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to(device)

print("Generating fine-tuned output...\n")
outputs = model.generate(
    input_ids=input_ids,
    max_length=60,         # Give it enough room to write a 2-line poem
    num_beams=5,           # Use beam search for higher quality text
    early_stopping=True,
    no_repeat_ngram_size=2 # Prevent the model from repeating the same words in a loop
)

# 4. Decode and print the result
final_doha = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("-" * 40)
print("Fine-Tuned Output:")
print(final_doha)
print("-" * 40)

1. Loading the newly trained model...


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model loaded and ready!

Input Prompt: Generate Hindi Doha | Theme: भक्ति | Context: ईश्वर के प्रति समर्पण
Generating fine-tuned output...

----------------------------------------
Fine-Tuned Output:
<extra_id_0> , न । ॥  <extra_id_1> र ही ते ई 
----------------------------------------


In [7]:
model.save_pretrained("/kaggle/working/doha_mt5_model")
tokenizer.save_pretrained("/kaggle/working/doha_mt5_model")

('/kaggle/working/doha_mt5_model/tokenizer_config.json',
 '/kaggle/working/doha_mt5_model/tokenizer.json')